### Relatório Financeiro
1. escolha o seu relatório;
2. estratégia para cada relatório. 


### 1° que eu tenho que desenvolver é:
leitor
* lê todo o PDF [x]
* lê páginas específicas [x]
* lê apenas o que é tabela [ ]

extrator: 
* Extrair de acordo com a página [x]
* Extrair já direto do site [ ]

layout
( forçar layout de alguma maneira )
* delimitar a seção com início e fim [x] ( cabeçalho deve ser injetado na pipeline )
* formatar datas para um mesmo padrão [ ]
* delimitação por categoria


* para forçar um código, eu tenho que colocar um dicionário com regex esperados. como por exemplo:
    Forte
    (PD < 5%)
    Estágio 1 → linha válida
    Estágio 2 → linha válida

### Extração determinística

1. Percorrer páginas sequencialmente
2. Detectar START ( onde estará texto "x" )
3. Começar a capturar linhas
4. Continuar até encontrar END ( onde estará texto "y" )
5. Se END não aparecer → falha 

### Formatação da tabela
Aqui é onde você vai formatar a tabela com um layout, adaptando a entrada para todos os RIs

In [1]:
# Nossas opções:
# Nubank, Inter, Nubank, BB, Santander, Itaú ( todos tem textos selecionáveis ). Resultado é definição de uma janela

# Ingerir PDF
import pdfplumber
import pandas as pd
import re
import numpy as np

In [15]:
with pdfplumber.open("Demonstrações Financeiras 3T25.pdf") as pdf:
    page = pdf.pages[17]   # página 18 (índice começa em 0)
    text = page.extract_text()

# Ajuste para manter a linha forte no padrão. 
# text = re.sub(
#     r"(Forte)\s+(.*)",   
#     r"\1\n\2",        
#     text
# )

# text = re.sub(
#     r"(.*?)(\(PD < 5%\))(.*)",  
#     r"\1\3\n\2",         
#     text,
#     count=1                  
# )

# Remove espaços em branco extras de cada linha ( pré-processamento )
text = "\n".join(line.strip() for line in text.splitlines())

# Define início ( start ) e fim ( end ) do trecho a ser extraído. 
start = text.find("c) Provisão para perdas de crédito - por qualidade de crédito vs. estágios")
end = text.find("Total", start)

trecho = text[start:end]
print(trecho)

# divide em linhas para facilitar a criação do dataframe
linhas = [l.strip() for l in trecho.splitlines() if l.strip()]


c) Provisão para perdas de crédito - por qualidade de crédito vs. estágios
30/09/2025 31/12/2024
Provisão Índice de Provisão Índice de
Exposição Exposição
% para perdas % cobertura % para perdas % cobertura
bruta bruta
de crédito (%) de crédito (%)
Forte (PD < 5%) 9.202.509 45,9% 193.814 5,9% 2,1% 6.644.920 45,5% 126.401 5,3% 1,9%
Estágio 1 9.202.497 100,0% 193.814 100,0% 2,1% 6.628.863 99,8% 126.147 99,8% 1,9%
Estágio 2 12 – – – – 16.057 0,2% 254 0,2% 1,6%
Satisfatório
6.063.886 30,3% 509.759 15,4% 8,7% 4.304.062 29,4% 324.830 13,6% 7,5%
(5% ≤ PD ≤ 20%)
Estágio 1 5.837.116 96,3% 490.321 96,1% 8,4% 4.170.990 96,9% 315.603 97,2% 7,6%
Estágio 2 226.770 3,7% 19.438 3,9% 8,6% 133.072 3,1% 9.227 2,8% 6,9%
Risco maior
4.761.732 23,8% 2.601.176 78,7% 54,6% 3.670.330 25,1% 1.938.295 81,1% 52,8%
(PD > 20%)
Estágio 1 990.988 20,8% 191.305 7,4% 19,4% 1.049.233 28,6% 229.234 11,8% 21,8%
Estágio 2 1.863.300 39,1% 796.156 30,6% 42,7% 1.228.767 33,5% 436.515 22,5% 35,5%
Estágio 3 1.907.444 40,1% 1.61

### Limpeza de dados

In [17]:
# dicionário de regras para criação das colunas do Dataframe. 
# É uma verificação para garantir que sempre haverá o título de cada coluna no local correto
QUALIDADE_RULES = {
    "Forte": "Forte (PD < 5%)",
    "Satisfatório": "Satisfatório (5% <= PD <= 20%)",
    "Risco maior": "Risco maior (PD > 20%)",
}

COLUMN_MAP = {
    "30/09/2025": {
        "Exposição bruta": "e25",
        "% Exposição": "p25",
        "Provisão para perdas de crédito": "pr25",
        "% Provisão": "pp25",
        "Índice de cobertura": "i25",
    },
    "31/12/2024": {
        "Exposição bruta": "e24",
        "% Exposição": "p24",
        "Provisão para perdas de crédito": "pr24",
        "% Provisão": "pp24",
        "Índice de cobertura": "i24",
    },
}

In [18]:
# Encontra padrões númericos das linhas a fim de criar regras para a criação do DataFrame
pattern = re.compile(
    r'^\s*(?P<label>Estágio\s+\d+)?\s*'
    r'(?P<e25>[\d\.]+)\s+(?P<p25>[\d,]+%|—)\s+'
    r'(?P<pr25>[\d\.]+|—)\s+(?P<pp25>[\d,]+%|—)\s+'
    r'(?P<i25>[\d,]+%|—)\s+'
    r'(?P<e24>[\d\.]+)\s+(?P<p24>[\d,]+%|—)\s+'
    r'(?P<pr24>[\d\.]+|—)\s+(?P<pp24>[\d,]+%|—)\s+'
    r'(?P<i24>[\d,]+%|—)\s*$'
)

In [19]:
rows = []
qualidade_atual = None

for linha in linhas:

    # Atualiza contexto (qualidade) e verifica se o que está contido no nosso dicionário bate com a regra.
    # como nem todas as linhas possuem a palavra "Forte", por exemplo, é feito essa verificação. É para agrupar parágrafos
    

    for key, value in QUALIDADE_RULES.items():
        if linha.startswith(key):
            qualidade_atual = value

        # REMOVE o texto da qualidade da linha
        # Ex: "Forte (PD < 5%)   9.202.509 ..." → "9.202.509 ..."
            linha = re.sub(r'^.*?(?=\d)', '', linha).strip()

            break


    # Ignora até identificar uma qualidade
    if qualidade_atual is None:
        continue
    
    # Normalização, a fim de que, mesmo linhas vazias ainda sejam consideradas
    # line = line.replace("–", "—").replace("-", "—")

    # Busca verificar se o trecho do dados é de fato, um dado. 
    match = pattern.search(linha)
    if not match:
        continue
    
    # Converte o grupo selecionado em dicionário
    d = match.groupdict()

    # Linha base ( cria colunas que não vem de regex automaticamente )
    row = {
        "Qualidade": qualidade_atual,
        "Estágio": d.get("label") or "Total",
    }

    # Preenche colunas dinamicamente
    for data_ref, fields in COLUMN_MAP.items():
        for col_name, regex_key in fields.items():
            row[f"{col_name} ({data_ref})"] = d.get(regex_key)

    rows.append(row)

In [20]:
print(type(pd))

<class 'module'>


In [ ]:
# dataframe final
df = pd.DataFrame(rows)

# transforma os dados com linha em dados vazios mesmo
df = df.replace({"—":  np.nan, "-": np.nan})

# Ordenação opcional para ficar igual ao relatório
stage_order = {"Total": 0, "Estágio 1": 1, "Estágio 2": 2, "Estágio 3": 3}
df["_ord"] = df["Estágio"].map(stage_order).fillna(99)
df = df.sort_values(["Qualidade", "_ord"]).drop(columns="_ord")

df = df.set_index(["Qualidade", "Estágio"])

df

Exposição bruta (30/09/2025)  \
Qualidade                      Estágio                                  
Forte (PD < 5%)                Total                        6.913.897   
                               Estágio 1                    6.913.873   
Risco maior (PD > 20%)         Estágio 1                    1.143.576   
                               Estágio 2                    1.521.384   
                               Estágio 3                    1.515.734   
Satisfatório (5% <= PD <= 20%) Estágio 1                    4.932.583   
                               Estágio 2                      241.980   

                                         % Exposição (30/09/2025)  \
Qualidade                      Estágio                              
Forte (PD < 5%)                Total                        42,5%   
                               Estágio 1                   100,0%   
Risco maior (PD > 20%)         Estágio 1                    27,4%   
                               Estágio 2                    36,4%   
                               Estágio 3                    36,3%   
Satisfatório (5% <= PD <= 20%) Estágio 1                    95,4%   
                               Estágio 2                     4,7%   

                                         Provisão para perdas de crédito (30/09/2025)  \
Qualidade                      Estágio                                                  
Forte (PD < 5%)                Total                                          139.509   
                               Estágio 1                                      139.509   
Risco maior (PD > 20%)         Estágio 1                                      195.635   
                               Estágio 2                                      644.450   
                               Estágio 3                                    1.382.983   
Satisfatório (5% <= PD <= 20%) Estágio 1                                      384.535   
                               Estágio 2                                       16.212   

                                         % Provisão (30/09/2025)  \
Qualidade                      Estágio                             
Forte (PD < 5%)                Total                        5,0%   
                               Estágio 1                  100,0%   
Risco maior (PD > 20%)         Estágio 1                    8,8%   
                               Estágio 2                   29,0%   
                               Estágio 3                   62,3%   
Satisfatório (5% <= PD <= 20%) Estágio 1                   96,0%   
                               Estágio 2                    4,1%   

                                         Índice de cobertura (30/09/2025)  \
Qualidade                      Estágio                                      
Forte (PD < 5%)                Total                                 2,0%   
                               Estágio 1                             2,0%   
Risco maior (PD > 20%)         Estágio 1                            17,2%   
                               Estágio 2                            42,4%   
                               Estágio 3                            91,2%   
Satisfatório (5% <= PD <= 20%) Estágio 1                             7,8%   
                               Estágio 2                             6,7%   

                                         Exposição bruta (31/12/2024)  \
Qualidade                      Estágio                                  
Forte (PD < 5%)                Total                        6.644.920   
                               Estágio 1                    6.628.863   
Risco maior (PD > 20%)         Estágio 1                    1.049.233   
                               Estágio 2                    1.228.767   
                               Estágio 3                    1.392.330   
Satisfatório (5% <= PD <= 20%) Estágio 1                    4.170.990   
                               Estágio 2                      133.07